In [3]:
######################################-----------------#####################################################
######################################-OTHER FUNCTIONS-#####################################################
######################################-----------------#####################################################
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns
###########################################################################################################

def counts_plots(f_gt, flag, pop_map):

#open gt file generated by write_gt function and load it as dataframe
    filegt = f_gt    
    raw_data = pd.read_table(filegt, header=[0], sep='\t', low_memory=False) #low_memery=False because individuals from POP_OUT could have missing data coded as nan and sometimes they are imported as mixed dtypes

#read the pop_map linking individuals to species/populations/groups
    df = pd.read_table(pop_map, sep='\t', header=None)
    df.rename(columns={0: 'samples', 1: 'species'}, inplace=True)
    inds = df['samples'].to_list()
    
#filter the genotype data including only selected individuals
    data = raw_data.dropna(axis=0, how='any', subset=inds)

#list of SNPeff annotated fitness effects
    effects = ['MODIFIER', 'LOW', 'MODERATE', 'HIGH']

#make summary counts of 0s, 1s, and 2s per individual using the flag selected by the user
    eff_list = []
    for ind in inds:
        eff_list_ind = []
        for eff in effects:
            eff_list_ind += data[(data['flag'].str.contains(flag))&(data['effect']==eff)][ind].value_counts().reindex([0,1,2], fill_value=0).to_list()
        eff_list.append(eff_list_ind)

    zip_eff_list = list(map(list, zip(*eff_list)))

    eff_list_colNAmes = ['MF_0','MF_1','MF_2','LO_0','LO_1','LO_2','MD_0','MD_1','MD_2','HI_0','HI_1','HI_2']

    i = 0
    for name in eff_list_colNAmes:
        df[name] = zip_eff_list[i]
        i += 1
#write counts to text file
    df.to_csv(filegt.replace('.gt', '.'+flag+'.counts'), sep='\t', header=True, index=False)

#plot the counts per species/populations/groups
    fig, axs = plt.subplots(3,4, figsize=(18, 18), facecolor='w', edgecolor='k', sharey=False)
    axs = axs.ravel()

    cat = ['MF_0','LO_0','MD_0','HI_0',
           'MF_1','LO_1','MD_1','HI_1',
           'MF_2','LO_2','MD_2','HI_2']

    j = 0
    for i in cat:
        sns.stripplot(data=df, y=i, x='species', ax=axs[j], hue='species')
        j+=1

    fig.savefig(filegt.replace('.gt', '.'+flag+'.pdf'), bbox_inches='tight')

############################################################################################################

def counts_plots_allele_lowCov(f_gt, flag, pop1, pop2, pop_map):

#open gt file generated by write_gt function and load it as dataframe
    filegt = f_gt    
    raw_data = pd.read_table(filegt, header=[0], sep='\t', low_memory=False) #low_memery=False because individuals from POP_OUT could have missing data coded as nan and sometimes they are imported as mixed dtypes

#read the pop_map linking individuals to species/populations/groups
    df = pd.read_table(pop_map, sep='\t', header=None)
    df.rename(columns={0: 'samples', 1: 'species'}, inplace=True)
    inds = df['samples'].to_list()
    
#filter the genotype data including only selected individuals
    data = raw_data.dropna(axis=0, how='any', subset=inds)

#list of SNPeff annotated fitness effects
    effects = ['MODIFIER', 'LOW', 'MODERATE', 'HIGH']

#make summary counts of 0s, 1s, and 2s per individual using the flag selected by the user
    eff_list = []
    for ind in inds:
        eff_list_ind = []
        for eff in effects:
            eff_list_ind += data[(data['flag'].str.contains(flag))&(data['effect']==eff)][ind].value_counts().reindex(['0','1'], fill_value=0).to_list()
        eff_list.append(eff_list_ind)

    zip_eff_list = list(map(list, zip(*eff_list)))

    eff_list_colNAmes = ['MF_0','MF_1','LO_0','LO_1','MD_0','MD_1','HI_0','HI_1']

    i = 0
    for name in eff_list_colNAmes:
        df[name] = zip_eff_list[i]
        i += 1

#write counts to text file    
    df.to_csv(filegt.replace('.covRandom1', '.'+flag+'lowCov.counts'), sep='\t', header=True, index=False)

#plot the counts per species/populations/groups
    fig, axs = plt.subplots(2,4, figsize=(18, 18), facecolor='w', edgecolor='k', sharey=False)
    axs = axs.ravel()


    cat = ['MF_0','LO_0','MD_0','HI_0',
           'MF_1','LO_1','MD_1','HI_1']

    j = 0
    for i in cat:
        sns.stripplot(data=df, y=i, x='species', ax=axs[j], hue='species')
        j+=1

    fig.savefig(filegt.replace('.covRandom1', '.'+flag+'lowCov.pdf'), bbox_inches='tight')

############################################################################################################

def freqChange_stat(f_gt, flag, popx, popy, nameX, nameY):
    
#open gt file generated by write_gt function and load it as dataframe
    filegt = f_gt
    raw_data = pd.read_table(filegt, header=[0], sep='\t', low_memory=False) #low_memery=False because individuals from POP_OUT could have missing data coded as nan and sometimes they are imported as mixed dtypes

    inds = popx+popy
    data = raw_data.dropna(axis=0, how='any', subset=inds)

    effects = ['MODIFIER', 'LOW', 'MODERATE', 'HIGH']

#Rxy estimates and plot of results in a inline table (also saved as Fig)
    Rxy = []
    AvgFreq_x = []
    AvgFreq_y = []

    for eff in effects:
        x = popx
        y = popy

        nDt = data[(data['flag'].str.contains(flag))&(data['effect']==eff)][inds].astype(int)
        
        nDt['freq_x'] = nDt[x].sum(axis=1)/(len(x)*2)
        nDt['freq_y'] = nDt[y].sum(axis=1)/(len(y)*2)
       
        
        nDt['Lx_Noty'] = (nDt['freq_x'])*(1-(nDt[y].sum(axis=1)/(len(y)*2)))
        nDt['Ly_Notx'] = (nDt['freq_y'])*(1-(nDt[x].sum(axis=1)/(len(x)*2)))
        Rxy.append(round(nDt['Lx_Noty'].sum()/nDt['Ly_Notx'].sum(), 3))
        
        AvgFreq_x.append(round(nDt['freq_x'].mean(), 3))
        AvgFreq_y.append(round(nDt['freq_y'].mean(), 3))
        
        tab1 = [Rxy, AvgFreq_x, AvgFreq_y]
        
    fig, ax = plt.subplots(figsize=(6, 2))
    fig.patch.set_visible(False)
    ax.axis('off')
    ax.axis('tight')

    ax.table(cellText=tab1, colLabels=effects, rowLabels=['Rxy', 'Freq_x', 'Frex_y'], loc='center')
    
    footer_text = 'x: '+nameX+', y: '+nameY
    plt.figtext(0.95, 0.05, footer_text, horizontalalignment='left', size=8)

    fig.tight_layout()
    plt.show()


In [ ]:
###COUNT AND PLOT GENOTYPE COUNTS FROM GENOLOADER GT FILE###
###USE counts_plots_allele_lowCov for gt with one resampled allele per genotype###

###Set the corresponding population/species for each sample in pop1+pop2
#The file loaded as pop_map must be a text file with one 'sample<tab>pop' attribution per line
#Example:
#ind1	sp1
#ind2	sp1
#ind3	sp2
#ind4	sp2
pop_map = 'pop_map'

#Choose the polarization flag you want to use: eg., 'unfolded' for all flags containing 'unfolded'. Be more specific if you need eg., 'unfoldedFixDer' for using only 'unfoldedFixDer1' and 'unfoldedFixDer2'
#Check carefully write_gt/get_ancestral_allele function to know more about the polarization flags
flag = 'unfolded'

###GENOLOADER generated gt file with ind names as in pop1, pop2, pop_out
f_gt = 'file1.ann.POP_OUT.gt'

###Make genotype counts plot for MODIFIER, LOW, MODERATE, and HIGH variants
counts_plots(f_gt, flag, pop_map)

In [ ]:
###STATS from GiNOLOADER GT FILE###

###Set ingroups samples lists
popx = ['sample1','sample2','sample3', '...'] 
popy = ['sample4','sample5','sample6', '...'] 

name_popx = 'name1'
name_popy = 'name2'

#Choose the polarization flag you want to use: eg., 'unfolded' for all flags containing 'unfolded'. Be more specific if you need eg., 'unfoldedFixDer' for using only 'unfoldedFixDer1' and 'unfoldedFixDer2'
#Check carefully get_ancestral_allele function to know more about the polarization flags - hope to provide graphical examples soon!
flag = 'unfolded'

###GiNOLOADER generated gt file with ind names as in pop1, pop2, pop_out
f_gt = 'file1.ann.POP_OUT.gt'

###Make genotype counts plot for MODIFIER, LOW, MODERATE, and HIGH variants
freqChange_stat(f_gt, flag, popx, popy, name_popx, name_popy)